# Week 6 – Video test (both)

**Done when:** a 1-minute traffic video gives a clean list of cars and plates with **no duplicates**.

- **Vision (Doğukan):** tracking with ByteTrack, every 3rd frame (`app/video_pipeline.py`, `app/bytetrack_anpr.yaml`)
- **Builder (Cemre):** plate voting over frames + each car saved once (`ocr/plate_vote.py`, app Video tab)

**Before you start, put in Drive:**
| Drive path | What |
|---|---|
| `ANPR/models/plate_best.pt` | plate detector (plate_v2 if it was better) |
| `ANPR/models/cls_best.pt` | vehicle-type classifier |
| `ANPR/videos/traffic.mp4` | a ~1 minute traffic video where plates are visible (film one from a footbridge / car park exit) |

Runtime → **T4 GPU**. Run cells one by one.

In [ ]:
!pip install -q ultralytics "lap>=0.5.12" paddlepaddle paddleocr

In [ ]:
import os
if not os.path.exists('anpr-project-en'):
    !git clone -q https://github.com/cemregonenc/anpr-project-en.git
else:
    !cd anpr-project-en && git pull -q
os.chdir('/content/anpr-project-en')
print(os.getcwd())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/ANPR'
!mkdir -p models && cp "$D/models/plate_best.pt" models/ ; cp "$D/models/cls_best.pt" models/
VIDEO = f'{D}/videos/traffic.mp4'
if not os.path.exists(VIDEO):
    # fallback: a 7-second Turkish street sample (Apache-2.0, github.com/Semihocakli/turkish-plate-recognition-w-yolov8-onnx-to-engine-cpp)
    # it already has boxes drawn on it, so use it only to check that the code runs
    VIDEO = '/content/sample.mp4'
    !wget -q -O /content/sample.mp4 https://github.com/Semihocakli/turkish-plate-recognition-w-yolov8-onnx-to-engine-cpp/raw/main/yolov8-tensorrt/result/detection-yolov8-detect/araba.mp4
print('video:', VIDEO, '| models:', os.listdir('models'))

## 1) Run the video pipeline (every 3rd frame)

In [ ]:
import sys, time, pandas as pd
sys.path.insert(0, 'app')
from pipeline import ANPRPipeline
from video_pipeline import VideoANPR, save_csv
pipe = ANPRPipeline()
print('warnings:', pipe.warnings)
t0 = time.time()
rows = VideoANPR(pipe, stride=3).process(VIDEO, out_video=f'{D}/results/week6_annotated.mp4')
print(f'{time.time() - t0:.0f} s')
df = pd.DataFrame(rows); df

## 2) Checks for the 'done when' line

In [ ]:
plates = df[df['plate_valid'] == True]['plate']
dups = plates[plates.duplicated()].unique()
print('cars:', len(df), '| with a valid plate:', len(plates), '| duplicate plates:', len(dups), list(dups))
print('DONE WHEN (no duplicates):', 'PASSED' if len(dups) == 0 else 'NOT YET')
save_csv(rows, f'{D}/results/week6_video_results.csv')
print('Now watch ANPR/results/week6_annotated.mp4 and count by eye: real cars vs rows in the table.')

## 3) Write down (for `docs/experiments.md` and the report)
| Video | Length | Real cars (by eye) | Rows in table | Duplicates | Plates correct |
|---|---|---|---|---|---|
| traffic.mp4 | | | | | |

If one car got two rows without a plate: raise `track_buffer` in `app/bytetrack_anpr.yaml` or lower the frame step.
If many tiny fake cars appear: raise `min_frames` (default 3) or the detection confidence.